# WEL01 · M09 사후 검토 지표 재계산과 보정 후보 선택 (학생 실습용)

- 본 노트북은 **미션9(M09) 전용 학습자 실습용** 노트북입니다.
- 원자료 생성·참고 구현·상황판 소스가 노트북에 내장되어 있어 별도의 `.md` 파일을 업로드할 필요가 없습니다.
- 위에서부터 첫 실행 셀, 준비 셀, `STEP 01`~`05`의 `TODO`를 순서대로 완성한 뒤 마무리 셀을 실행하면 `WEL01/outputs/M09/WEL01_M09_result.html`이 생성됩니다.

In [ ]:
from pathlib import Path
import base64,zlib,json,re,html,hashlib,io,time,ast,copy
import numpy as np,pandas as pd
from IPython.display import display,HTML

PROJECT_ID='WEL01'
ROLE='student'
WORKDIR=Path('/content') if Path('/content').is_dir() else Path.cwd()
PROJECT_HOME=WORKDIR/PROJECT_ID
OUTDIR=PROJECT_HOME/'outputs'
SOURCE_BUNDLE=None
DASHBOARD_ASSETS=None
REFERENCE_CACHE={}
REFERENCE_EXECUTION_SERIAL=0

def _sha(value):
    return hashlib.sha256(json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()).hexdigest()

EMBEDDED_BUNDLE_B64 = '...'
EMBEDDED_ASSETS_B64 = 'eNqNOl1z20aSfwVxNgERgwgpiaIEHuLzeu0938ZOKvLePsgqGQSGJCIQ4GJASjqGW/Za2XIib61u177IieSS7+zE3kvqFH8qVd6X/BwC+g/XPTMAAYrM5cEiMNPT3dPf3XD/lEXpKf3UO/26v1Gkzr87XlOv+4FNgiKsDOq+vdlvm0HT8fRSrW5aa83A73q2/iZpNGYb8zXLd/1Af7NcnanPWrWG74V6udLZeLesVSrS2cAxXVW+7Ie+tGR6VPrNR7JK4aFISeA0BvVuGPqe6nidbqiGZCM0A2L2GRLHawFIKED6VjegQKfjO15IgoHmmpt+N+zbDu3Ao94MHLuGf4ohacNKSIrAWLftUb08O9PZkNqO1zY3CiW13AiUGrwVW8RptoDZUqnXGmg0JJ2iZ/b62SuWq7P1uUVxxfWWE5Jax7RtFFF5AZAudjZqHZ86oeN7Og0da22zFvodEJTpOk0PLuk2YN0MwtpkahINA99rpteou761xoSIqiCMSkqyJCHNmVJnI4NAiCePYN2xwxbSequGQi0ybnSXNMJaokqpJJXLGeRlFFIZcNe48nXYlKjvOrb05lx93qxWxEYxMG2nS/Uqgo5kFQag1Q6ozwuz8jrB6bIJJlEEbSKkgZsrOZGTcn1h3kytqjRjzy4mlMUiB8kKsW267nQZ4jVdxyOpxrU5IYYiKquC8rQAHBjqo2Fw6ZVS0cwAhDSLogELErvlSonpgWySeuCv9wVrc4vzi9WFLPEZJE5CsNkiiMdCfFppgbRTklKr3B/BI60xZmcridJA5Ki2UoZhqTWTPc2ZTDW8kIOczUCW5ydAUsvvkOQq89U5u7yY8/hGo1Gx6yOTKaHJzKUmk1hGBjVGAqk00NAzf46vjrxUq4KfJl47Mw+kuOc2zQ73iowGuUt0TI+4OQVmeOfOe9K07SopkXHTXsz6xeKIVt0HC24z8gMtNOsuKVIr8MH2/B4JGq6/rpvd0GdmItQ3V0HeGGx/ZMWu2aFETx6y7pq3nEHYUkO7P4Fti8yRhRPOnTBdFbGJ3ZrZHdE9fz0wO4Ax724LjVJjdnIQG4QY/KUw0L2wVbRajmsXSI94Sg5Do9owG3WQh++7dTNItdxwyQbTFiqLh0Ngpk11i2AEr+F+EVnS8U9qMDPcYAS2JLyNWeEERdbnrarVGFNkJaPICpfJCDVLOv3s9sLE6CcwCyXNsXDRJmHgWD8r+zQCiRlFIoo0t3JbGtEhM2SRWDkdlrL2MDsiLNVTi+MiNL3NdUiXJGsRARognOiGxO6PRSTh45V6tVy1wHVcP+xnrFAYL+M7H1YzXldmknD9Zj9rZp2AMJZqUxgci8sZV5lFV6nlPGmgeX5IspxlEzfLVgFhyOAUWK7p/lzLmMtYBo99oRl2aRL8ZiqV6lyVM7vOydV91x5oDR8kNXaHJGAuVK2F6kkJUbNH/r+qQnBd+gn7xdzPYsI/t4ntmIVRLlpAwSn9NIdlSxQReqfaZlpITYFgMT5XO2VKpZQQOs/8pLIklyqquWyUy3ogxsHgqnfV0xwPSqKuhcGIZgV9Mi8uVKbY2CCHRGrNJfXrzBxLoEwsORAen8d9LQdi+Tbp5+NpAGXHiSpkrNCYzmMqxRRgU9j8BNZ4suGq0hvOBrHHwVrq2II9gdtpzFz1TqmnPsY2oFBQjPf6cpcSLE0dK5TBTgGvZBnrjmf769rvzr9fKq+e++DyhYu/xspGMoFijxgltWM28afhuBDfDTk5iS5MjbNBYG4WKooG225BlhWxGzpgXU1qLK+IBeIahdBsqpZLWUeA/PCdDcP2rW4brEezoE0IyXmX4BuCKzWnUYAjyoZmuSall802MeAdlxHLG4YBvkcaoB0bYHDpHLdEYwnu6TUZlALhJOwGnrQxENyYnc6IapOEguQvNy/aBRk203tw3RgE7mY7PVmV+UK6D5pmm/ALm4nyYRv+apSEZ0NgA7yGAFqskF2zTlyAjP73KP50V4q2Hw+fbQlwoEs8yMWAjvcPsup1XVe1tLZDKahfSai2TcdjZPEBsAnvAzycvQQVYFURRKnBQrLIQRJU3KOZoq56ja7H7EwCi/Bc30QMbaL2TLdLFKEuc5q6ZBPom1orIA3jtx+9L3Y/qH9MrBDeCx5Zl37p+vXC8r8ufXBZo0w/TmOzwNDzq84oK2o/3OwQHbXgQuhHdt79mPqePFAQfcKZgZzBuwVAawWlBrK+4rQJ3IzZOjIQkJ6/lmGA86aokHFKyiBzW9ry16+gKxZ4p6OKOKlCC0CTe6NrZewgWyeOrB4XGRB7StdbBBhmy/gkK2oY8NeAgTBikIOC86bVKqwZ74VB1hjCljCENQVEgBiS3RAqZ0YpWcDNVLNQ5XEq+ASEgECBbwWS35Cyl4PnEUMjuDWEE/wloD0jWF5bqcGJHI+24LFnGAY+fPIJPqXOeQYt/vje3Xj/SNZRwYAYAWSfqUc+M24Tit5TFDamSMgARWWQuy3uKjVUTCoQ3FZqXI3JIgLk9M2rrUTZa2pvXAxcxRwMBDJ2Vwi9XqoRFVfqyeWVcdqM6Qxp2INCoJAQpIaFJUqT0GUeb1dqwiFTcyjUVQeDJUQ8BwTGwZT6pOAi2m8RiIBx4kLAr4MjtCFDTINWBkoNgwSAQZVgkXPYEcBOQSxnbs7lIppjWb32i76ldQIfNXjRHkg/vpJwRUSr9L3ldwM6iLf3hodbuBZvP4wP9qQPN8MWSATejv/zM2n49HD47PU1Ls9WOQ19oROCRtlqJ12k0GzAHXxlGous5YXf4YudeO9B/OUT6fjuk3jreym+vxP9923GxTdbEH2l+C787LKFw6PhswMp+vvj6M9Hgi1gCU7eiz9/AQfj/V0JVG+bVHAbPXotDV98G73cAuzR9q1o+6GGC/GDQ4C4Hn+9H+9fl+IXe/F/fSp9eHZpSTq+vRMfgBM853TviUNyGtmxXsuYIL5CuMAuMF1VVNYCjN7FUdYos1VKmLHBebYmo1HCr2aboQlmo3EbWgIbSbIkXzldTiAzAm3NCLFfW7py/kNQ6PgR6NDtJRyGFWZUuQTmJAEQ5YobCIUKh8KehZkNBSPx4I0OQHhSdHgIUmE6uLnPtPTFc4YDojk0c+dYAGLmFH+1g8aCe2CB56DwDgfDwz0pevY8fvD8msLqBaq5xAw8EvxL2HYTT8tUUWMyytZXIKrMK5ReiOXKpfeNHM68kDIHlAEnxmslYhuMTarx90JgvPcGf/zkEyHFYFnmd7r4K3lFAYKW27UJLXCw1CywuaXZ7MObXbAFaHfZOmt7GfsdjblxCxobrNcS9Mwib34mM4ifqEzG4TlKlqMNzhVb8D2rZXpQGbLKUhSHKWSNF421JNwldRe0kbxe4lFOWJYMDhEfbElAF27kQcU2AQb8JN6/zWFqiEeDFhbDvW0wYgaQw6Oj5QKug4G+M1t6z0hUAnr0mmGLY4BLYPnA74DQxWKGZYbtBMjp0xkQppeRIXRURMtuoI4yRcbuEQGwc7o8+MMv+pfMsIV+UMgwqo7xCd70Lpj72OoApHBNGXNWxgsUQpliBnZVqiXVTIqEwo1IQfCiZqgrE/y/w9IJdtGpcfz4Cn6jZ0fDw7/++Cq6tRUf3BARkXluLgJK8b0tiLgY4TCKJZgZmcS6KSQIi0wJXXwzF5FmU7v5akcE9Oj2daQpxU//R5pnNpKVBMeBiYM9nBMVF5dDSV1U0q2P0F2T9fkxpjmIkmmApvCMW5M5Hj6/Hh9ySe1DutmShkeH0YP9UQJIvpbwcky8AGpECZiTlTEf3zo4vrl/fHdPGr56Pfz+UGXp6/Zj0E5SdUnR0zvx3pYqJX7EVpGRaPt5tP1DvJ/wcnx3N946ir+8o8kjctz/Wc+XVirppu+x6MN8JAdi5M/n5JJspUZg9kjO7WVcwRzOuJKwQJTig+vx/UdoFbCX8820Y7nGO9lsIbIaEMsPbA3biGtqP61Y9Ez1ogpgPT2mcnVfgDBNgg5E6xD2GqM3lV1V7TqXiEm7AWuCqC4aX7VNTA9HJHJaPkhc90LKEPKiJ4dgBbtovuA9kAF/fAX1CkCJMkHUJtD45ASHV8/bJe6mYuy222awOc2Z+O4U20yLpfjmjeN7T4ALtEtRLYujqny8uxXf/zZpXTGqJaUAxqrKtZMnRm6Knpn3NRHmTpwZxRNZzRcCE/Bz5Lx4kFk6iZ7eAhPfHT79R/zFTry1l1R38T/uRH/bE9WdPAGVAONFafz8lqxe7rbrEO0tHNKaUCQ3IIZ6FlkF6fqeTRUt9C/g6KYwq4ymBpBLqdE3Pb9tupurFhYqOrjpa8YKZLubN4DG3vEXu1J8axdEAg08dJIpINSIACGCWhawGZg2WUUqDtT4nrWpy9Hfvh8eAa/7r9HAUAABcU3UyGrDDMD2bV6c6PKFsx9Jw8Od+OXd6NEBgDl0DV3DdF0gubfFiNz/FvW791hW2Twq3QerjR9CMXsCrNuBspKsor2vtqkui7J+ePh9vH0gHf/ls+GrA6lNR4DmxgQ4KJLhtgyOZ1cGI7QxfHUUfQ2B6uB29Oc70Tc3GBgbXAODicgOH4NFRjsg3i/vxE+/4+Iyu7YTrkLwcezVOoGWioC8nnGpQsERPdgD3CJTyWpotjtA2SYhOA6xU1Dcf7mX7qdfVVJUe1vxD0CtUV5tQjuy2umArMtwtT0IzNJbkEEtaBscdn3B7vHNHSiupOOvtqB0TWygRwJIxXCXb66j+x/ffQwbiS1GP3B5Wy1iraW3hq7kKZw/2AEe4cB30d+fcGwdk1K8whjEn14wr0IIMI7U0iBlY3MGBiKnpRoJ8JOOgWObJRIWluUTtgeuNtnUcGNkXPCWtSU2r+JXxR2oYleyM4flNZW5zwrOHvjwRgM+AgeqYu5XymgUwZ0S5xFQ9+cGDnz+gIFNQeeCboMkERLKAs8Q19NaJi2sKWcKwst7yjvlUmnk02XltPxWfmLhMUj57bff4Gc0h170QtJkp8/00qMzSu5YHcoz8BH5TKF3Ro53b8kgfIzvt+Ldh7Ki98ZDEburyphVBhPidmdUWIrQhO3dKut8IRCxHjFNGfx4miMQZlqGwL0p+YEXdLyQiO9dl6I/PsYGV4x10iZ9jK22bxN31eyCuwe05XSU3ISJbKKel2X8/x9gYMjNqgs5FKcYzG/5UtLQi7SKLRBLtKtdj4EmFpUCcs9nVdOKgkNkYSlkc0U5cccTTHO4CdLIQMq/vSiSIMYSSDYSH3Gwnp+POI5vPI++/g69XIqPHoCvQW2FWT9T0kB6ElVNZoaQS1Y4dohf3okeHEB9IKIlI8SLt4fRI4hgh3+N73+amSbk1c9sopZKQWMxhL79du41ScX9pBClIZ1iJGxvSkmeJFCesyDzb++dKHKhS2KY4RdH6j7oc2z+l+MsmQ2PEYbTYxdl28oApybptBvqJD4wyc+KEEaIBD+Tjt+erU0bpjD4n749huyk8mBlCS/OMY/JOQeggcWGrJyLdFrRbvKuvo2mjlvYh7ebGoAb8I89m25oJNVYnjBYTDLv2j443oXif3cn+vyOBCvR548SPvIXAYzKICshzpCYaICOpkgDdibLIs8ZZBhYSJMZS7xJyBhZgVA7DW38RJHXGACkEQa/10LpeelXEuTK6NsjNmfM1OaDzJwRXCt69h/QCr28jiUGeCFLuV/8CdogbKJzH8D+7eL5360uXTl75bzRP9kRjAZnelLzqskcSh+NpNRsuarnq1coqwIaYumri+FQ0m3OKlN6iUENJ8hgMfhZzjFKNeefKjXn9OnEWOrZtkmp1dM5H81M+Bwc7tWnjLHHRnvOpKmemOSxz6GjQSyfXTsrYk6LJHITExFLMN028FslFM6a568XlJr4vugko6LR98VkvAIPv++CR5/1nDbLAhcCs03Y553pO30hNK3Tpa1CH0Wgw32Ad/w/OfYlqp9gpRiqLELqMmO8GPrFcN0vmgnyYgOxU+zDptiKlleckShOUQbpPJ/xU8996asrg/SuAwX+XPVODf4PT6yS7w=='

def _decode_b64(payload):
    return json.loads(zlib.decompress(base64.b64decode(payload, validate=True)))

def load_embedded_bundle():
    global SOURCE_BUNDLE, DASHBOARD_ASSETS, REFERENCE_CACHE, REFERENCE_EXECUTION_SERIAL
    bundle = _decode_b64(EMBEDDED_BUNDLE_B64)
    loaded_assets = _decode_b64(EMBEDDED_ASSETS_B64)
    if bundle.get('projectId') != PROJECT_ID or bundle.get('version') != 'wel01-integrated-2':
        raise ValueError('프로젝트·버전 불일치')
    if bundle.get('fingerprint') != _sha({k: v for k, v in bundle.items() if k != 'fingerprint'}):
        raise ValueError('원자료 지문 불일치')
    if bundle.get('assetFingerprint') != _sha(loaded_assets):
        raise ValueError('화면 소스 지문 불일치')
    for m_id, item in bundle['missions'].items():
        if len(item['stages']) != 5:
            raise ValueError(m_id + ' 5단계 누락')
        if sum(s['minutes'] for s in item['stages']) != item['minutes']:
            raise ValueError(m_id + ' 시간 불일치')
        if hashlib.sha256(item['referenceCode'].encode()).hexdigest() != item['referenceCodeSha256']:
            raise ValueError(m_id + ' 참고 코드 지문 불일치')
    namespace = {}
    exec(compile(bundle['runtimeSource'], 'WEL01_reference_runtime.py', 'exec'), namespace)
    bundle['_reference_runner'] = namespace['_reference_table']
    SOURCE_BUNDLE, DASHBOARD_ASSETS, REFERENCE_CACHE = bundle, loaded_assets, {}
    REFERENCE_EXECUTION_SERIAL = 0
    print(f'WEL01 M09 내장 교육 자료 연결 완료 (5단계 · {bundle["missions"]["M09"]["minutes"]}분)')
    return bundle

def _connected():
    if SOURCE_BUNDLE is None:raise RuntimeError('먼저 첫 실행 셀을 실행해 내장 교육 자료를 로드하세요.')

def _parameter_code(mid,code,parameters):
    allowed=SOURCE_BUNDLE['missions'][mid]['parameters']
    if set(parameters)-set(allowed):raise ValueError('정의되지 않은 변경 파라미터')
    if mid=='M02':
        value=parameters['FLATLINE_RUN']
        if not isinstance(value,int) or value<1:raise ValueError('고착 분은 양의 정수')
        code=code.replace('def flatline(series, run=8):',f'def flatline(series, run={value}):')
    elif mid=='M05':
        value=float(parameters['SCORE_THRESHOLD'])
        if not 0<=value<=1:raise ValueError('문턱은 0~1')
        code=code.replace('eng = (test["score"] >= 0.5).values',f'eng = (test["score"] >= {value!r}).values')
    elif mid=='M09':
        value=float(parameters['MIN_RECALL'])
        if not 0<=value<=1:raise ValueError('재현율 하한은 0~1')
        code=code.replace('(g["재현율"] >= 0.90)',f'(g["재현율"] >= {value!r})')
    inserts={}
    for node in ast.parse(code).body:
        if isinstance(node,ast.Assign):
            names=[]
            for target in node.targets:
                names.extend(x.id for x in ast.walk(target) if isinstance(x,ast.Name))
            updates=[f'{key} = {parameters[key]!r}' for key in names if key in parameters and key not in ['MIN_RECALL','SCORE_THRESHOLD','FLATLINE_RUN']]
            if updates:inserts[node.end_lineno]='\n'.join(updates)
    lines=code.splitlines()
    for pos,value in sorted(inserts.items(),reverse=True):lines.insert(pos,value)
    return '\n'.join(lines)+'\n'

def source_table(mid,parameters=None,refresh=False):
    global REFERENCE_EXECUTION_SERIAL
    _connected();item=SOURCE_BUNDLE['missions'][mid]
    p=dict(item['parameters']);p.update(parameters or {})
    key=(mid,_sha(p))
    if refresh or key not in REFERENCE_CACHE:
        tuned=None
        if mid=='M10':
            # 현재 M9 실행 결과를 우선 사용  아직 실행하지 않았으면 기본 M9를 실행한다.
            matches=[(k,v) for k,v in REFERENCE_CACHE.items() if k[0]=='M09']
            if not matches:
                source_table('M09');matches=[(k,v) for k,v in REFERENCE_CACHE.items() if k[0]=='M09']
            m9=max(matches,key=lambda x:x[1][1]['executionOrder'])[1][1]['selected']
            tuned={'heat_temp':float(m9['th_heat']),'cold_temp':float(m9['th_cold']),
                'min_persist':int(m9['persist']),'wind_corr':float(m9['corr'])}
        code=_parameter_code(mid,item['referenceCode'],p)
        raw,meta=SOURCE_BUNDLE['_reference_runner'](mid,code,tuned)
        REFERENCE_EXECUTION_SERIAL+=1
        meta['executionOrder']=REFERENCE_EXECUTION_SERIAL
        meta.update(parameters=p,md_fingerprint=SOURCE_BUNDLE['fingerprint'],reference_name=mid+' 교육용 참고 구현')
        REFERENCE_CACHE[key]=(raw,meta)
        if mid=='M09':
            for stale in [k for k in REFERENCE_CACHE if k[0]=='M10']:del REFERENCE_CACHE[stale]
    raw,meta=REFERENCE_CACHE[key]
    out=raw.copy(deep=True);out.attrs['reference_meta']=copy.deepcopy(meta)
    return out

def TODO(mission,step,instruction):
    raise NotImplementedError(f'{mission} STEP {step}: TODO를 실제 pandas 코드로 완성하세요. {instruction}')

def _rows(df):
    return json.loads(df.to_json(orient='records',force_ascii=False,date_format='iso'))

def finish_mission(mid,raw,stages,stage_specs):
    _connected();canonical=SOURCE_BUNDLE['missions'][mid]['stages']
    if stage_specs!=canonical:raise ValueError(mid+' 내장 기준과 노트북 STEP 정의 불일치')
    if len(stages)!=5 or not isinstance(raw,pd.DataFrame) or raw.empty:raise ValueError('5단계와 원자료 필요')
    expected=list(raw.columns);original_ids=raw['원행ID'].tolist();report=[];cumulative=[]
    for k,(df,spec) in enumerate(zip(stages,stage_specs),1):
        if not isinstance(df,pd.DataFrame):raise TypeError(f'{mid} STEP{k}: DataFrame 필요')
        col=spec['outputColumn'];cumulative.append(col)
        if df['원행ID'].tolist()!=original_ids or not df.index.equals(raw.index):
            raise ValueError(f'{mid} STEP{k}: 행 누락·재정렬·중복이 발생했습니다.')
        if any(x not in df.columns for x in expected+cumulative):raise ValueError(f'{mid} STEP{k}: 필수 열 누락')
        if not df[expected].equals(raw[expected]):raise ValueError(f'{mid} STEP{k}: 원자료를 변경했습니다.')
        if df.columns.duplicated().any():raise ValueError('열 이름 중복')
        preview=df[['원행ID']+cumulative]
        report.append(dict(step=k,title=spec['title'],goal=spec['goal'],outputColumn=col,
            learnerHtml=spec.get('learnerHtml',''),rowCount=len(df),columns=list(preview.columns),rows=_rows(preview),minutes=spec['minutes']))
    location=OUTDIR/mid;location.mkdir(parents=True,exist_ok=True);stem=f'WEL01_{mid}'
    stages[-1].to_csv(location/(stem+'_analysis.csv'),index=False,encoding='utf-8-sig')
    meta=copy.deepcopy(raw.attrs['reference_meta']);plots=meta.pop('plots',[])
    details=dict(projectId='WEL01',mission=mid,originalRows=len(raw),codeSteps=report,
        referenceMeta=meta,mdFingerprint=SOURCE_BUNDLE['fingerprint'],
        structuralValidation='원행과 원본값·누적 열 검증 완료  코드의 의미·교관 PASS 판정과 별개')
    (location/(stem+'_code_report.json')).write_text(json.dumps(details,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    config=dict(projectId='WEL01',mission=mid,title=SOURCE_BUNDLE['missions'][mid]['title'],
        scenario=SOURCE_BUNDLE['missions'][mid]['scenario'],hours=SOURCE_BUNDLE['missions'][mid]['minutes']/60,
        stages=report,sourceColumns=list(raw.columns),sourceRows=_rows(raw),meta=meta,plots=plots,
        fingerprint=SOURCE_BUNDLE['fingerprint'],role=ROLE)
    safe=json.dumps(config,ensure_ascii=False,default=str).replace('<','\\u003c')
    document='<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>'+stem+'</title><style>'+DASHBOARD_ASSETS['css']+'</style></head><body><div id="app"></div><script>window.WEL01_CONFIG='+safe+';</script><script>'+DASHBOARD_ASSETS['js']+'</script></body></html>'
    path=location/(stem+'_result.html');path.write_text(document,encoding='utf-8')
    if not globals().get('HEADLESS_VALIDATION',False):
        display(HTML('<iframe title="'+stem+'" sandbox="allow-scripts allow-downloads" style="width:100%;height:1150px;border:0" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
    print('HTML:',path);print('행 보존·누적 열 검사 완료  최종 PASS는 교관 판정')
    return path

load_embedded_bundle()

## M09 사후 검토 지표 재계산과 보정 후보 선택

사후 검토(AAR)를 위해 96개 설정 후보의 TP·FP·TN·FN에서 지표를 다시 계산합니다. 오경보율과 재현율 조건을 통과한 후보만 정해진 순서로 검토하세요. 같은 자료에 다시 맞춘 결과라는 한계와 남은 오류의 의미도 함께 적어야 합니다.

실습 시간 180분. 준비 셀 → STEP별 설명과 코드 → 마무리 셀 순서로 실행합니다.


In [ ]:
MID = 'M09'
PARAMETERS = {'MIN_RECALL': 0.9}
raw = source_table(MID, parameters=PARAMETERS, refresh=True)
print("입력 표의 행·열 수:", raw.shape)
display(raw.head(6))


### M09 STEP 01 — 정상 구간을 분모로 오경보율 다시 계산하기

<h4>이 단계에서 해결할 일</h4><p>사후 검토에서는 기존 지표 이름을 믿고 옮기기보다 원래 건수에서 직접 계산해야 합니다. 96개 보정 후보마다 정상 기준 구간 중 오경보가 발생한 비율인 FAR를 구합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m09_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>raw</code>, 완성한 결과 표는 <code>step1</code>입니다. 새로 만들 열은 <code>FAR_재계산</code>이며 배정 시간은 36분입니다. 노트북의 <code>M09 STEP 01</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>FP</code> — 정상 기준 구간에 경보가 발생한 건수입니다.</li><li><code>TN</code> — 정상 기준 구간에 경보가 발생하지 않은 건수입니다.</li></ul><h4>수행 순서</h4><ol><li>raw에서 각 후보의 FP와 TN을 확인합니다.</li><li>정상 구간 수 FP+TN을 분모로 계산합니다.</li><li>FP를 분모로 나눈 값을 FAR_재계산 열에 저장하여 step1을 만듭니다. 분모가 0이면 NaN으로 남깁니다.</li><li>0~1 수치로 저장하고 보고서에서만 100을 곱해 %로 표시합니다. 후보별 분자·분모를 기록해 계산 근거를 남깁니다.</li></ol><h4>결과를 읽는 방법</h4><p>FAR는 정상 구간 가운데 경보가 울린 비율입니다. 경보 중 틀린 비율 FP/(TP+FP)과 다릅니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"FP": 10, "TN": 90}</code></td><td><code>0.1</code></td><td>정상 100건 중 10건입니다.</td></tr><tr><td><code>{"FP": 0, "TN": 0}</code></td><td><code>NaN 또는 빈값</code></td><td>분모가 없어 NaN입니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>정상 구간이 없으면 FAR를 평가할 수 없습니다. NaN을 0으로 바꿔 우수 후보처럼 만들지 않습니다.</p>

In [ ]:
# M09 STEP 01 — 정상 구간을 분모로 오경보율 다시 계산하기
step1 = TODO(MID, 1, '정상 구간을 분모로 오경보율 다시 계산하기')
display(step1[["원행ID", 'FAR_재계산']].head(10))


### M09 STEP 02 — 위험 구간을 분모로 재현율 다시 계산하기

<h4>이 단계에서 해결할 일</h4><p>오경보를 낮춘 후보라도 위험을 놓치면 선택하기 어렵습니다. 각 후보가 찾아야 할 위험 구간 가운데 실제 경보를 낸 비율을 같은 건수 표에서 계산합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m09_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step1</code>, 완성한 결과 표는 <code>step2</code>입니다. 새로 만들 열은 <code>재현율_재계산</code>이며 배정 시간은 36분입니다. 노트북의 <code>M09 STEP 02</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>TP</code> — 위험 기준 구간에서 경보가 발생한 건수입니다.</li><li><code>FN</code> — 위험 기준 구간에서 경보가 발생하지 않은 건수입니다.</li></ul><h4>수행 순서</h4><ol><li>step1에서 TP와 FN을 읽습니다.</li><li>위험 기준 구간 전체 수 TP+FN을 계산합니다.</li><li>TP/(TP+FN)을 재현율_재계산 열에 저장하여 step2를 만듭니다. 분모 0은 NaN으로 남깁니다.</li><li>FAR와 나란히 확인하고 재현율이 낮은 후보의 FN이 얼마나 되는지 기록합니다.</li></ol><h4>결과를 읽는 방법</h4><p>0.90은 위험 기준 구간의 90%를 찾아냈다는 뜻입니다. 경보의 90%가 맞았다는 뜻은 아닙니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"TP": 90, "FN": 10}</code></td><td><code>0.9</code></td><td>위험 100건 중 90건을 찾았습니다.</td></tr><tr><td><code>{"TP": 0, "FN": 0}</code></td><td><code>NaN 또는 빈값</code></td><td>계산할 위험 구간이 없습니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>위험 구간이 없는 후보는 재현율을 확인할 수 없습니다. 1로 채우지 않습니다.</p>

In [ ]:
# M09 STEP 02 — 위험 구간을 분모로 재현율 다시 계산하기
step2 = TODO(MID, 2, '위험 구간을 분모로 재현율 다시 계산하기')
display(step2[["원행ID", '재현율_재계산']].head(10))


### M09 STEP 03 — TP·FP·FN에서 F1 점수 직접 계산하기

<h4>이 단계에서 해결할 일</h4><p>정밀도와 재현율을 함께 보는 F1을 기존 반올림 지표에서 옮기지 않고 원래 건수로 계산합니다. 이 값은 뒤에서 후보 순위를 정할 때 사용됩니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m09_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step2</code>, 완성한 결과 표는 <code>step3</code>입니다. 새로 만들 열은 <code>F1_재계산</code>이며 배정 시간은 36분입니다. 노트북의 <code>M09 STEP 03</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>TP·FP·FN</code> — 앞 단계와 같은 후보의 원래 건수입니다. F1 분모에는 TN이 들어가지 않습니다.</li></ul><h4>수행 순서</h4><ol><li>step2에서 TP·FP·FN을 확인합니다.</li><li>분자는 2×TP, 분모는 2×TP+FP+FN으로 계산합니다.</li><li>분자/분모를 F1_재계산에 넣고 step3에 저장합니다. 분모가 0이면 NaN으로 둡니다.</li><li>0~1 원래 수치를 유지합니다. 순위를 매기기 전에 소수 둘째 자리 등으로 반올림하지 않습니다.</li></ol><h4>결과를 읽는 방법</h4><p>F1은 정밀도와 재현율의 균형을 나타내는 점수입니다. 전체 정확도와 같은 값이 아닙니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"TP": 40, "FP": 10, "FN": 10}</code></td><td><code>0.8</code></td><td>80/(80+10+10)=0.8입니다.</td></tr><tr><td><code>{"TP": 0, "FP": 0, "FN": 0}</code></td><td><code>NaN 또는 빈값</code></td><td>분모가 0이므로 NaN입니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>이미 반올림된 정밀도·재현율로 다시 계산하면 순위가 달라질 수 있습니다. 원래 건수를 사용합니다.</p>

In [ ]:
# M09 STEP 03 — TP·FP·FN에서 F1 점수 직접 계산하기
step3 = TODO(MID, 3, 'TP·FP·FN에서 F1 점수 직접 계산하기')
display(step3[["원행ID", 'F1_재계산']].head(10))


### M09 STEP 04 — 오경보율과 재현율 조건을 모두 만족하는 후보 고르기

<h4>이 단계에서 해결할 일</h4><p>점수가 높다는 이유만으로 선택하지 않고 최소 조건을 먼저 확인합니다. FAR 15% 이하와 현재 재현율 하한을 동시에 만족한 후보만 순위 대상에 포함합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m09_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step3</code>, 완성한 결과 표는 <code>step4</code>입니다. 새로 만들 열은 <code>제약_충족</code>이며 배정 시간은 36분입니다. 노트북의 <code>M09 STEP 04</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>FAR_재계산·재현율_재계산</code> — 앞 단계의 0~1 수치입니다.</li><li><code>PARAMETERS["MIN_RECALL"]</code> — 기본 재현율 하한은 0.90입니다. 변경 실험의 실제 값과 구분합니다.</li></ul><h4>수행 순서</h4><ol><li>step3에서 FAR와 재현율, 현재 하한을 확인합니다.</li><li>FAR≤0.15와 재현율≥MIN_RECALL을 각각 비교합니다.</li><li>두 조건을 모두 만족한 행만 제약_충족=True로 하여 step4에 저장합니다. NaN이 있는 행은 조건 충족으로 보지 않습니다.</li><li>통과 후보 수를 기록합니다. 0개이면 조건 안에서 선택할 수 없다고 적고 임의로 조건을 완화하지 않습니다.</li></ol><h4>결과를 읽는 방법</h4><p>True는 이 두 수치 조건의 통과 표시이며 독립된 새 자료에서 성능이 입증되었다는 뜻은 아닙니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"FAR_재계산": 0.15, "재현율_재계산": 0.9, "MIN_RECALL": 0.9}</code></td><td><code>True</code></td><td>두 경계값을 모두 포함합니다.</td></tr><tr><td><code>{"FAR_재계산": 0.151, "재현율_재계산": 0.99, "MIN_RECALL": 0.9}</code></td><td><code>False</code></td><td>FAR 조건이 미달입니다.</td></tr><tr><td><code>{"FAR_재계산": 0.1, "재현율_재계산": 0.899, "MIN_RECALL": 0.9}</code></td><td><code>False</code></td><td>재현율 조건이 미달입니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>0.15를 15와 비교하지 않습니다. 기본 조건 결과와 하한을 바꾼 실험 결과를 따로 남깁니다.</p>

In [ ]:
# M09 STEP 04 — 오경보율과 재현율 조건을 모두 만족하는 후보 고르기
step4 = TODO(MID, 4, '오경보율과 재현율 조건을 모두 만족하는 후보 고르기')
display(step4[["원행ID", '제약_충족']].head(10))


### M09 STEP 05 — 조건을 통과한 후보에만 선택 순위 붙이기

<h4>이 단계에서 해결할 일</h4><p>최소 조건을 통과한 후보 중 남은 오류가 적은 설정부터 검토합니다. 원자료의 재발이라는 열 이름은 이번 코드에서 시간상 재발을 뜻하지 않으므로 의미를 바로 읽어야 합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m09_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step4</code>, 완성한 결과 표는 <code>step5</code>입니다. 새로 만들 열은 <code>선택_순위</code>이며 배정 시간은 36분입니다. 노트북의 <code>M09 STEP 05</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>제약_충족</code> — STEP 4에서 두 조건을 모두 통과한 표시입니다.</li><li><code>재발</code> — 초기와 보정 후 모두 남아 있는 동일 병사·원인의 오류 키 수입니다. 독립된 후속 기간의 재발 횟수가 아닙니다.</li><li><code>F1_재계산·재현율_재계산</code> — 앞 단계의 반올림하지 않은 값입니다.</li></ul><h4>수행 순서</h4><ol><li>step4에서 제약_충족=True인 행만 순위 계산용으로 선택합니다.</li><li>재발은 작은 순서, 같으면 F1은 큰 순서, 다시 같으면 재현율은 큰 순서로 정렬합니다. 세 값이 모두 같으면 원래 행 순서를 유지합니다.</li><li>정렬한 후보에 1부터 순위를 붙인 뒤 원래 인덱스에 다시 맞춰 선택_순위 열로 저장합니다. 조건 미달 행은 빈값(pd.NA), 열 형식은 nullable Int64로 하여 step5를 만듭니다.</li><li>1위의 원행ID·설정·세 정렬값을 기록합니다. 통과 후보가 없으면 전부 빈값으로 두고 선택 불가 사유를 적습니다.</li></ol><h4>결과를 읽는 방법</h4><p>같은 합성 자료에서 탐색한 후보의 우선 검토 순위입니다. 새 부대·새 기간에 대한 일반화 성능 순위가 아닙니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"후보A": [true, 2, 0.91, 0.92], "후보B": [true, 1, 0.9, 0.91]}</code></td><td><code>B=1위, A=2위</code></td><td>배열은 충족 여부·잔존 오류 키·F1·재현율 순입니다. 오류 키가 우선합니다.</td></tr><tr><td><code>{"제약_충족": false}</code></td><td><code>NaN 또는 빈값</code></td><td>조건 미달 후보는 순위가 빈값입니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>빈 순위는 의도된 결과입니다. 0위로 채우거나 미달 후보에 후순위를 붙이지 않습니다. 원래 96행 순서를 바꾸지 않습니다.</p>

In [ ]:
# M09 STEP 05 — 조건을 통과한 후보에만 선택 순위 붙이기
step5 = TODO(MID, 5, '조건을 통과한 후보에만 선택 순위 붙이기')
display(step5[["원행ID", '선택_순위']].head(10))


### 마무리: 결과 화면 생성과 기록 저장

아래 셀을 실행하면 다섯 단계의 실제 계산 결과가 HTML로 저장됩니다. 왼쪽 STEP 메뉴로 결과를 확인하고 관찰 기록을 입력하세요. 기록 JSON을 내려받아야 입력한 글을 별도로 보관할 수 있습니다. 구조 검사 완료는 교관의 최종 합격 판정이 아닙니다.


In [ ]:
STAGE_SPECS = SOURCE_BUNDLE["missions"][MID]["stages"]
result_html = finish_mission(MID, raw, [step1, step2, step3, step4, step5], STAGE_SPECS)
